# Publication checks for the leakage-safe RF + BGWO SQL-injection pipeline

This notebook implements the applicable items from the publication checklist:

1. Dataset cleaning audit with class counts before/after.
2. Environment, package, and hardware details.
3. Exact-duplicate and template-leakage audit.
4. Repeated leakage-safe experiments across five seeds (baseline RF, RF+BGWO, RF+BGWO top-13).
5. Text baselines (word and character TF-IDF with Logistic Regression and Linear SVM).
6. Mean, standard deviation, and 95% confidence intervals across seeds.
7. Top-13 feature stability across seeds.
8. A group-aware (template-grouped) stress test with no template overlap between train and test.
9. False positives/negatives by heuristic attack family (proxy, because no family labels exist).
10. A checklist summary mapping each item to its evidence.

All feature selection and hyperparameters are frozen using training/validation data only. The test partitions are used only for final scoring. External/cross-dataset validation is not possible without an external dataset and is marked as not available.


In [1]:

from pathlib import Path
import json
import os
import platform
import re
import sys
import time
import warnings
from collections import Counter
from contextlib import redirect_stdout
from importlib.metadata import version as pkg_version
from io import StringIO

import joblib
import numpy as np
import pandas as pd
import psutil
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("Modified_SQL_Dataset_Strip.csv")
ARTIFACT_DIR = Path("publication_checks_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

MAX_FEATURES = 512
NGRAM_RANGE = (1, 2)
TOP_K = 13
SEEDS = [42, 43, 44, 45, 46]
BGWO_POPULATION = 8
BGWO_ITERATIONS = 15
FITNESS_RF_TREES = 60
RF_TREES = 200
CHAR_MAX_FEATURES = 5000


## 1. Dataset audit: cleaning and class counts before/after

The raw CSV is read without modification, then the exact cleaning steps used by the main notebook are applied. Exact duplicates on `(Query, Label)` are removed. Label mapping is 0 = benign, 1 = SQL injection.


In [2]:

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")

raw = pd.read_csv(DATA_PATH)
required = {"Query", "Label"}
missing_columns = required - set(raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

raw_rows = len(raw)
raw_counts = raw["Label"].value_counts().sort_index().to_dict()

df = raw.loc[:, ["Query", "Label"]].dropna().copy()
rows_after_dropna = len(df)
df["Query"] = df["Query"].astype(str).str.strip()
df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
df = df.dropna(subset=["Label"])
df["Label"] = df["Label"].astype(int)
df = df[df["Query"].str.len() > 0]
rows_after_empty = len(df)
df = df.drop_duplicates(subset=["Query", "Label"]).reset_index(drop=True)
rows_after_dedupe = len(df)

labels = set(df["Label"].unique())
if not labels.issubset({0, 1}) or len(labels) != 2:
    raise ValueError(f"Expected binary labels 0/1; found {sorted(labels)}")

counts_after = df["Label"].value_counts().sort_index()
audit = {
    "raw_rows": int(raw_rows),
    "raw_class_counts": {str(k): int(v) for k, v in raw_counts.items()},
    "rows_after_dropna": int(rows_after_dropna),
    "rows_after_empty_query_removal": int(rows_after_empty),
    "rows_after_duplicate_removal": int(rows_after_dedupe),
    "duplicates_removed": int(rows_after_empty - rows_after_dedupe),
    "final_class_counts": {str(k): int(v) for k, v in counts_after.items()},
    "exact_duplicate_queries_remaining": int(df["Query"].duplicated().sum()),
    "query_label_conflicts": int(df.groupby("Query")["Label"].nunique().gt(1).sum()),
    "label_mapping": {"0": "Benign", "1": "SQL injection"},
}
display(pd.Series(audit, name="value").to_frame())
with open(ARTIFACT_DIR / "dataset_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)


,value
raw_rows,22916
raw_class_counts,"{'0': 11584, '1': 11332}"
rows_after_dropna,22916
rows_after_empty_query_removal,22916
rows_after_duplicate_removal,22859
duplicates_removed,57
final_class_counts,"{'0': 11581, '1': 11278}"
exact_duplicate_queries_remaining,0
query_label_conflicts,0
label_mapping,"{'0': 'Benign', '1': 'SQL injection'}"


## 2. Environment details

Runtime versions and hardware details are recorded so the experiment can be reproduced and the latency/accuracy context reported correctly.


In [3]:

def package_versions():
    packages = [
        "scikit-learn", "pandas", "numpy", "scipy", "joblib",
        "psutil", "nbformat", "nbclient", "ipykernel",
    ]
    versions = {}
    for package in packages:
        try:
            versions[package] = pkg_version(package)
        except Exception:
            versions[package] = "not installed"
    return versions

sklearn_report = StringIO()
try:
    from sklearn import show_versions
    with redirect_stdout(sklearn_report):
        show_versions()
except Exception as exc:
    sklearn_report.write(f"show_versions failed: {exc}")

env = {
    "platform": platform.platform(),
    "processor": platform.processor(),
    "machine": platform.machine(),
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "cpu_count_logical": os.cpu_count(),
    "ram_total_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
    "packages": package_versions(),
}
display(pd.Series(env, name="value").to_frame())
print(sklearn_report.getvalue())
with open(ARTIFACT_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
with open(ARTIFACT_DIR / "sklearn_versions.txt", "w", encoding="utf-8") as f:
    f.write(sklearn_report.getvalue())


,value
platform,Windows-11-10.0.26200-SP0
processor,"Intel64 Family 6 Model 142 Stepping 10, Genuin..."
machine,AMD64
python,3.13.15
python_executable,C:\Users\user\AppData\Local\Programs\Python\Py...
cpu_count_logical,8
ram_total_gb,7.88
packages,"{'scikit-learn': '1.9.1', 'pandas': '2.3.3', '..."



System:
    python: 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
executable: C:\Users\user\AppData\Local\Programs\Python\Python313\python.exe
   machine: Windows-11-10.0.26200-SP0

Python dependencies:
      sklearn: 1.9.1
          pip: 26.2.1
   setuptools: 84.0.0
        numpy: 2.5.3
        scipy: 1.18.1
       Cython: None
       pandas: 2.3.3
   matplotlib: 3.11.2
       joblib: 1.6.0
threadpoolctl: 3.7.0
     narwhals: 2.26.0

Built with OpenMP: True

threadpoolctl info:
       user_api: blas
   internal_api: openblas
    num_threads: 8
         prefix: libscipy_openblas
       filepath: C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy.libs\libscipy_openblas64_-ed4f167a5330424524f45258e7ca2c8d.dll
        version: 0.3.34.106.0
threading_layer: pthreads
   architecture: Haswell

       user_api: blas
   internal_api: openblas
    num_threads: 8
         prefix: libscipy_openblas
       filepath: C:\Users\user\App

## 3. Duplicate and template-leakage audit

Exact duplicates are already removed. This section goes further by normalizing each query into a template (quoted literals and numbers replaced by placeholders) and measuring how often a random-split test row shares a template with the training partition. A high overlap indicates that random-split metrics may benefit from template memorization; the group-aware experiment in Section 8 provides a stricter estimate.


In [4]:

def normalize_template(query):
    s = str(query).lower()
    s = re.sub(r"'(?:''|[^'])*'", " '?' ", s)
    s = re.sub(r'"(?:""|[^"])*"', ' "?" ', s)
    s = re.sub(r"\b\d+(?:\.\d+)?\b", " ? ", s)
    s = re.sub(r"[^a-z0-9_?\s;=<>()#\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["Template"] = df["Query"].map(normalize_template)
template_sizes = df.groupby("Template").size()
template_stats = {
    "unique_queries": int(df["Query"].nunique()),
    "unique_templates": int(df["Template"].nunique()),
    "templates_with_multiple_rows": int((template_sizes > 1).sum()),
    "max_template_group_size": int(template_sizes.max()),
    "median_template_group_size": float(template_sizes.median()),
    "rows_in_multi_row_templates": int(template_sizes[template_sizes > 1].sum()),
}

X_text_all = df["Query"].to_numpy()
y_all = df["Label"].to_numpy()
groups_all = df["Template"].to_numpy()

idx_tr, idx_te = train_test_split(
    np.arange(len(df)), test_size=0.30, random_state=SEED, stratify=y_all
)
train_templates = set(groups_all[idx_tr])
test_templates = set(groups_all[idx_te])
overlap_mask = np.isin(groups_all[idx_te], list(train_templates))
leakage = {
    **template_stats,
    "random_split_test_rows": int(len(idx_te)),
    "random_split_test_rows_with_template_in_train": int(overlap_mask.sum()),
    "random_split_test_template_overlap_pct": float(100 * overlap_mask.mean()),
    "random_split_test_injections_with_template_in_train_pct": float(
        100 * overlap_mask[y_all[idx_te] == 1].mean()
    ),
    "random_split_test_benign_with_template_in_train_pct": float(
        100 * overlap_mask[y_all[idx_te] == 0].mean()
    ),
    "exact_query_overlap_between_random_splits": int(
        len(set(X_text_all[idx_tr]) & set(X_text_all[idx_te]))
    ),
    "templates_in_both_random_splits": int(len(train_templates & test_templates)),
}
display(pd.Series(leakage, name="value").to_frame())
with open(ARTIFACT_DIR / "template_leakage.json", "w", encoding="utf-8") as f:
    json.dump(leakage, f, indent=2)


,value
unique_queries,22859.000000
unique_templates,17176.000000
templates_with_multiple_rows,1162.000000
max_template_group_size,521.000000
median_template_group_size,1.000000
rows_in_multi_row_templates,6845.000000
random_split_test_rows,6858.000000
random_split_test_rows_with_template_in_train,1929.000000
random_split_test_template_overlap_pct,28.127734
random_split_test_injections_with_template_in_train_pct,52.866430


## 4. Reusable leakage-safe pipeline

The helper functions are identical in spirit to the main notebook: TF-IDF is fitted on the fit subset, BGWO optimizes validation fitness, and the top-13 ranking uses an RF trained on the fit subset only. Seven models are evaluated per seed:

- Baseline RF on all word TF-IDF features.
- RF + BGWO on the BGWO-selected features.
- RF + BGWO top-13.
- Logistic Regression and Linear SVM on word TF-IDF.
- Logistic Regression and Linear SVM on character 3-5 gram TF-IDF.


In [5]:

def clean_texts(texts):
    return pd.Series(np.asarray(texts, dtype=object).reshape(-1)).fillna("").astype(str)


def mask_matrix(X, mask):
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return None
    return X[:, idx]


def make_fitness(X_fit, y_fit, X_val, y_val, n_estimators=60, seed=SEED):
    d = X_fit.shape[1]
    cache = {}

    def fitness(mask):
        mask = np.asarray(mask, dtype=bool)
        key = np.packbits(mask).tobytes()
        if key in cache:
            return cache[key]
        k = int(mask.sum())
        if k == 0:
            return 1.0
        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=seed,
            n_jobs=-1,
            class_weight="balanced_subsample",
            max_features="sqrt",
        )
        clf.fit(mask_matrix(X_fit, mask), y_fit)
        score = accuracy_score(y_val, clf.predict(mask_matrix(X_val, mask)))
        value = 0.99 * (1.0 - score) + 0.01 * (k / d)
        cache[key] = float(value)
        return float(value)

    return fitness


def binary_gwo(fitness, dim, population=8, iterations=15, seed=SEED):
    rng = np.random.default_rng(seed)
    positions = rng.uniform(-1, 1, size=(population, dim))
    best_mask = None
    best_score = np.inf
    history = []

    for t in range(iterations):
        binary = rng.random((population, dim)) < (1.0 / (1.0 + np.exp(-positions)))
        scores = np.array([fitness(row) for row in binary], dtype=float)
        order = np.argsort(scores)
        alpha = positions[order[0]].copy()
        beta = positions[order[1]].copy()
        delta = positions[order[2]].copy()
        if scores[order[0]] < best_score:
            best_score = float(scores[order[0]])
            best_mask = binary[order[0]].copy()
        history.append({
            "iteration": t + 1,
            "best_fitness": best_score,
            "selected_features": int(best_mask.sum()),
        })
        a = 2.0 - 2.0 * (t / max(iterations - 1, 1))
        r1 = rng.random((3, population, dim))
        r2 = rng.random((3, population, dim))
        leaders = [alpha, beta, delta]
        proposals = []
        for j in range(3):
            A = 2 * a * r1[j] - a
            C = 2 * r2[j]
            D = np.abs(C * leaders[j] - positions)
            proposals.append(leaders[j] - A * D)
        positions = np.clip(np.mean(proposals, axis=0), -6, 6)

    if best_mask is None or not best_mask.any():
        best_mask = np.zeros(dim, dtype=bool)
        best_mask[int(np.argmin(np.abs(positions.mean(axis=0))))] = True
        best_score = fitness(best_mask)
    return best_mask.astype(bool), best_score, pd.DataFrame(history)


def score_model(model_name, seed, model, X_eval, y_eval, score_kind="proba"):
    pred = model.predict(X_eval)
    if score_kind == "proba":
        scores = model.predict_proba(X_eval)[:, 1]
    else:
        scores = model.decision_function(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()
    return {
        "seed": seed,
        "model": model_name,
        "n_test": int(len(y_eval)),
        "n_features": int(X_eval.shape[1]),
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_eval, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, scores) if len(np.unique(y_eval)) == 2 else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 5. Run one complete experiment per seed

Each seed redoes the whole split, vectorization, BGWO optimization, top-13 ranking, and model fitting. This is the expensive part; progress is printed per seed. Seed 42 is retained for the FP/FN analysis in Section 9.


In [6]:

def run_seed(seed):
    t_start = time.perf_counter()

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X_text_all, y_all, test_size=0.30, random_state=seed, stratify=y_all
    )
    X_fit_text, X_val_text, y_fit, y_val = train_test_split(
        X_train_text, y_train, test_size=0.20, random_state=seed, stratify=y_train
    )

    word_vec = TfidfVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )
    X_fit = word_vec.fit_transform(clean_texts(X_fit_text))
    X_val = word_vec.transform(clean_texts(X_val_text))
    X_train = word_vec.transform(clean_texts(X_train_text))
    X_test = word_vec.transform(clean_texts(X_test_text))
    feature_names = np.asarray(word_vec.get_feature_names_out())

    fitness_fn = make_fitness(
        X_fit, y_fit, X_val, y_val,
        n_estimators=FITNESS_RF_TREES, seed=seed,
    )
    selected_mask, best_fitness, history = binary_gwo(
        fitness_fn,
        dim=X_fit.shape[1],
        population=BGWO_POPULATION,
        iterations=BGWO_ITERATIONS,
        seed=seed,
    )

    ranking_rf = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=seed, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    ranking_rf.fit(mask_matrix(X_fit, selected_mask), y_fit)
    selected_indices = np.flatnonzero(selected_mask)
    importance_order = np.argsort(ranking_rf.feature_importances_)[::-1][:TOP_K]
    top_indices = selected_indices[importance_order]
    top13_mask = np.zeros(len(feature_names), dtype=bool)
    top13_mask[top_indices] = True
    top13_names = feature_names[top_indices]

    baseline_rf = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=seed, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    bgwo_rf = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=seed, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    top13_rf = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=seed, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    baseline_rf.fit(X_train, y_train)
    bgwo_rf.fit(mask_matrix(X_train, selected_mask), y_train)
    top13_rf.fit(mask_matrix(X_train, top13_mask), y_train)

    lr_word = LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=seed
    )
    svm_word = LinearSVC(
        class_weight="balanced", random_state=seed, max_iter=5000
    )
    lr_word.fit(X_train, y_train)
    svm_word.fit(X_train, y_train)

    char_vec = TfidfVectorizer(
        lowercase=True,
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )
    char_vec.fit(clean_texts(X_fit_text))
    C_train = char_vec.transform(clean_texts(X_train_text))
    C_test = char_vec.transform(clean_texts(X_test_text))

    lr_char = LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=seed
    )
    svm_char = LinearSVC(
        class_weight="balanced", random_state=seed, max_iter=5000
    )
    lr_char.fit(C_train, y_train)
    svm_char.fit(C_train, y_train)

    rows = [
        score_model("Baseline RF (512 word TF-IDF)", seed, baseline_rf, X_test, y_test),
        score_model(
            "RF + BGWO (BGWO-selected features)", seed, bgwo_rf,
            mask_matrix(X_test, selected_mask), y_test,
        ),
        score_model(
            f"RF + BGWO top-{TOP_K}", seed, top13_rf,
            mask_matrix(X_test, top13_mask), y_test,
        ),
        score_model(
            "Logistic Regression (word TF-IDF)", seed, lr_word, X_test, y_test
        ),
        score_model(
            "Linear SVM (word TF-IDF)", seed, svm_word, X_test, y_test,
            score_kind="decision",
        ),
        score_model(
            "Logistic Regression (char 3-5 TF-IDF)", seed, lr_char, C_test, y_test
        ),
        score_model(
            "Linear SVM (char 3-5 TF-IDF)", seed, svm_char, C_test, y_test,
            score_kind="decision",
        ),
    ]

    result = {
        "rows": rows,
        "history": history,
        "top13_names": [str(name) for name in top13_names],
        "top13_importances": [float(v) for v in ranking_rf.feature_importances_[importance_order]],
        "selected_count": int(selected_mask.sum()),
        "total_features": int(len(selected_mask)),
        "best_fitness": float(best_fitness),
        "elapsed_seconds": time.perf_counter() - t_start,
    }
    if seed == SEED:
        result["audit"] = {
            "y_test": y_test.copy(),
            "text_test": np.asarray(X_test_text, dtype=object).copy(),
            "top13_pred": top13_rf.predict(mask_matrix(X_test, top13_mask)).copy(),
            "baseline_pred": baseline_rf.predict(X_test).copy(),
        }
    return result


all_rows = []
seed_top13 = {}
seed_meta = []
seed_audit = None

for seed in SEEDS:
    print(f"Running seed {seed} ...", flush=True)
    out = run_seed(seed)
    all_rows.extend(out["rows"])
    seed_top13[seed] = out["top13_names"]
    seed_meta.append({
        "seed": seed,
        "bgwo_selected_features": out["selected_count"],
        "total_features": out["total_features"],
        "bgwo_best_fitness": out["best_fitness"],
        "top13_features": out["top13_names"],
        "elapsed_seconds": out["elapsed_seconds"],
    })
    out["history"].assign(seed=seed).to_csv(
        ARTIFACT_DIR / f"bgwo_history_seed_{seed}.csv", index=False
    )
    if seed == SEED:
        seed_audit = out["audit"]
    print(
        f"  done in {out['elapsed_seconds']:.1f}s | "
        f"selected={out['selected_count']} | "
        f"top13={', '.join(out['top13_names'][:5])}...",
        flush=True,
    )

multiseed_metrics = pd.DataFrame(all_rows)
multiseed_metrics.to_csv(ARTIFACT_DIR / "multiseed_metrics.csv", index=False)
seed_meta_df = pd.DataFrame(seed_meta)
seed_meta_df.to_csv(ARTIFACT_DIR / "multiseed_run_metadata.csv", index=False)
display(seed_meta_df)


Running seed 42 ...


  done in 86.0s | selected=249 | top13=from, select, and, all select, all...


Running seed 43 ...


  done in 78.6s | selected=250 | top13=select, where, and, as, s from...


Running seed 44 ...


...

Running seed 45 ...


  done in 70.7s | selected=241 | top13=select, select from, and, or, all...


Running seed 46 ...


  done in 71.0s | selected=251 | top13=from, select, union, or, 5...


,seed,bgwo_selected_features,total_features,bgwo_best_fitness,top13_features,elapsed_seconds
0,42,249,512,0.005173,"[from, select, and, all select, all, 1 and, 1 ...",85.983668
1,43,250,512,0.006120,"[select, where, and, as, s from, delete, 1 whe...",78.579418
2,44,253,512,0.006797,"[1, from, select, all select, all, update, ins...",70.038009
3,45,241,512,0.005326,"[select, select from, and, or, all, union, whe...",70.728712
4,46,251,512,0.006758,"[from, select, union, or, 5, where, into, 1 se...",70.968232


## 6. Multi-seed results with 95% confidence intervals

Means and standard deviations are computed over the five seeds. The 95% interval uses the t distribution with n-1 degrees of freedom. Each seed has its own untouched test partition, which is the standard repeated-holdout design.


In [7]:

def mean_ci(series, confidence=0.95):
    values = np.asarray(series, dtype=float)
    n = len(values)
    mean = values.mean()
    if n < 2:
        return mean, np.nan, np.nan
    se = values.std(ddof=1) / np.sqrt(n)
    half = stats.t.ppf((1 + confidence) / 2, n - 1) * se
    return mean, mean - half, mean + half


metric_cols = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc"]
summary_rows = []
for model, group in multiseed_metrics.groupby("model", sort=False):
    row = {
        "model": model,
        "n_seeds": int(group["seed"].nunique()),
        "n_features_mean": float(group["n_features"].mean()),
        "n_features_min": int(group["n_features"].min()),
        "n_features_max": int(group["n_features"].max()),
    }
    for col in metric_cols:
        m, lo, hi = mean_ci(group[col])
        row[f"{col}_mean"] = m
        row[f"{col}_std"] = group[col].std(ddof=1)
        row[f"{col}_ci95_low"] = lo
        row[f"{col}_ci95_high"] = hi
    summary_rows.append(row)

multiseed_summary = pd.DataFrame(summary_rows)
multiseed_summary.to_csv(ARTIFACT_DIR / "multiseed_summary.csv", index=False)

display(multiseed_summary[[
    "model", "n_seeds", "n_features_mean", "n_features_min", "n_features_max",
    "accuracy_mean", "accuracy_std", "accuracy_ci95_low", "accuracy_ci95_high",
    "f1_mean", "f1_std", "f1_ci95_low", "f1_ci95_high",
    "roc_auc_mean", "roc_auc_std",
]])


,model,n_seeds,n_features_mean,n_features_min,n_features_max,accuracy_mean,accuracy_std,accuracy_ci95_low,accuracy_ci95_high,f1_mean,f1_std,f1_ci95_low,f1_ci95_high,roc_auc_mean,roc_auc_std
0,Baseline RF (512 word TF-IDF),5,512.0,512,512,0.998484,0.000561,0.997787,0.999180,0.998463,0.000568,0.997757,0.999169,0.999879,0.000207
1,RF + BGWO (BGWO-selected features),5,248.8,241,253,0.997696,0.000663,0.996872,0.998520,0.997666,0.000671,0.996833,0.998499,0.999616,0.000206
2,RF + BGWO top-13,5,13.0,13,13,0.988481,0.007274,0.979449,0.997512,0.988456,0.007247,0.979457,0.997455,0.998312,0.000588
3,Logistic Regression (word TF-IDF),5,512.0,512,512,0.996384,0.000466,0.995806,0.996962,0.996337,0.000471,0.995753,0.996921,0.999377,0.000255
4,Linear SVM (word TF-IDF),5,512.0,512,512,0.997813,0.000751,0.996881,0.998745,0.997784,0.000760,0.996840,0.998728,0.999319,0.000245
5,Logistic Regression (char 3-5 TF-IDF),5,5000.0,5000,5000,0.997638,0.000442,0.997089,0.998187,0.997603,0.000449,0.997046,0.998161,0.999623,0.000203
6,Linear SVM (char 3-5 TF-IDF),5,5000.0,5000,5000,0.998484,0.000394,0.997994,0.998973,0.998462,0.000399,0.997967,0.998958,0.999518,0.000193


## 7. Top-13 feature stability across seeds

If the same features recur across seeds, the top-13 conclusion is more robust. Pairwise Jaccard overlap summarizes set agreement.


In [8]:

feature_counts = Counter()
for names in seed_top13.values():
    feature_counts.update(names)

stability = pd.DataFrame([
    {
        "feature": feature,
        "seeds_present": count,
        "seeds_total": len(SEEDS),
        "frequency_pct": 100 * count / len(SEEDS),
    }
    for feature, count in feature_counts.most_common()
])
stability.to_csv(ARTIFACT_DIR / "top13_stability.csv", index=False)
display(stability)

jaccards = []
for i, s1 in enumerate(SEEDS):
    for s2 in SEEDS[i + 1:]:
        a = set(seed_top13[s1])
        b = set(seed_top13[s2])
        jaccards.append(len(a & b) / len(a | b))
print(
    f"Pairwise Jaccard overlap of top-13 sets: "
    f"mean={np.mean(jaccards):.3f}, min={np.min(jaccards):.3f}, max={np.max(jaccards):.3f}"
)


,feature,seeds_present,seeds_total,frequency_pct
0,select,5,5,100.0
1,where,4,5,80.0
2,from,3,5,60.0
3,and,3,5,60.0
4,all,3,5,60.0
5,1 select,3,5,60.0
6,5,3,5,60.0
7,union,3,5,60.0
8,1 or,3,5,60.0
9,else,3,5,60.0


Pairwise Jaccard overlap of top-13 sets: mean=0.233, min=0.083, max=0.444


## 8. Group-aware stress test

The dataset has no session/source IDs, so normalized query templates are used as groups. GroupShuffleSplit guarantees that no template is shared between the group-train and group-test partitions. The validation subset is also carved by groups from group-train. The whole pipeline (TF-IDF, BGWO, top-13 ranking, final RF models) is re-run inside this split. This is deliberately stricter than the random-split evaluation and quantifies template-memorization effects.


In [9]:

def valid_group_split(X_idx, y, groups, test_size, seed, attempts=100):
    for offset in range(attempts):
        splitter = GroupShuffleSplit(
            n_splits=1, test_size=test_size, random_state=seed + offset
        )
        a, b = next(splitter.split(X_idx, y, groups))
        if len(np.unique(y[a])) == 2 and len(np.unique(y[b])) == 2:
            return a, b
    raise RuntimeError("Could not produce a group split with both classes present.")


group_result = None
try:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    g_train_idx, g_test_idx = next(
        gss.split(np.arange(len(df)), y_all, groups_all)
    )
    if len(np.unique(y_all[g_train_idx])) < 2 or len(np.unique(y_all[g_test_idx])) < 2:
        g_train_idx, g_test_idx = valid_group_split(
            np.arange(len(df)), y_all, groups_all, 0.30, SEED
        )

    fit_pos, val_pos = valid_group_split(
        g_train_idx, y_all[g_train_idx], groups_all[g_train_idx], 0.20, SEED
    )
    fit_idx = g_train_idx[fit_pos]
    val_idx = g_train_idx[val_pos]

    overlap_train_test = int(
        len(set(groups_all[g_train_idx]) & set(groups_all[g_test_idx]))
    )
    overlap_fit_val = int(
        len(set(groups_all[fit_idx]) & set(groups_all[val_idx]))
    )

    gvec = TfidfVectorizer(
        lowercase=True, analyzer="word", token_pattern=r"(?u)\b\w+\b",
        ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES,
        sublinear_tf=True, dtype=np.float32,
    )
    G_fit = gvec.fit_transform(clean_texts(X_text_all[fit_idx]))
    G_val = gvec.transform(clean_texts(X_text_all[val_idx]))
    G_train = gvec.transform(clean_texts(X_text_all[g_train_idx]))
    G_test = gvec.transform(clean_texts(X_text_all[g_test_idx]))

    gfitness = make_fitness(
        G_fit, y_all[fit_idx], G_val, y_all[val_idx],
        n_estimators=FITNESS_RF_TREES, seed=SEED,
    )
    gmask, gfitness_best, ghistory = binary_gwo(
        gfitness, dim=G_fit.shape[1],
        population=BGWO_POPULATION, iterations=BGWO_ITERATIONS, seed=SEED,
    )
    ghistory.assign(seed=SEED).to_csv(
        ARTIFACT_DIR / "bgwo_history_group_split.csv", index=False
    )

    gnames = np.asarray(gvec.get_feature_names_out())
    grank = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    grank.fit(mask_matrix(G_fit, gmask), y_all[fit_idx])
    gsel = np.flatnonzero(gmask)
    gorder = np.argsort(grank.feature_importances_)[::-1][:TOP_K]
    gtop_idx = gsel[gorder]
    gtop_mask = np.zeros(len(gnames), dtype=bool)
    gtop_mask[gtop_idx] = True

    gbaseline = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    gbgwo = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    gtop = RandomForestClassifier(
        n_estimators=RF_TREES, random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample", max_features="sqrt",
    )
    gbaseline.fit(G_train, y_all[g_train_idx])
    gbgwo.fit(mask_matrix(G_train, gmask), y_all[g_train_idx])
    gtop.fit(mask_matrix(G_train, gtop_mask), y_all[g_train_idx])

    group_metrics = pd.DataFrame([
        score_model(
            "Baseline RF (group-aware split)", SEED, gbaseline,
            G_test, y_all[g_test_idx],
        ),
        score_model(
            f"RF + BGWO ({int(gmask.sum())} features, group-aware split)",
            SEED, gbgwo, mask_matrix(G_test, gmask), y_all[g_test_idx],
        ),
        score_model(
            f"RF + BGWO top-{TOP_K} (group-aware split)", SEED, gtop,
            mask_matrix(G_test, gtop_mask), y_all[g_test_idx],
        ),
    ])
    group_metrics.to_csv(ARTIFACT_DIR / "group_split_metrics.csv", index=False)
    group_result = {
        "group_train_rows": int(len(g_train_idx)),
        "group_test_rows": int(len(g_test_idx)),
        "fit_rows": int(len(fit_idx)),
        "validation_rows": int(len(val_idx)),
        "template_overlap_train_test": overlap_train_test,
        "template_overlap_fit_val": overlap_fit_val,
        "bgwo_selected_features": int(gmask.sum()),
        "bgwo_best_fitness": float(gfitness_best),
        "top13_features": [str(name) for name in gnames[gtop_idx]],
        "top13_importances": [float(v) for v in grank.feature_importances_[gorder]],
    }
    with open(ARTIFACT_DIR / "group_split_summary.json", "w", encoding="utf-8") as f:
        json.dump(group_result, f, indent=2, ensure_ascii=False)
    display(group_metrics)
    print(json.dumps(group_result, indent=2, ensure_ascii=False))
except Exception as exc:
    group_result = {"error": str(exc)}
    with open(ARTIFACT_DIR / "group_split_summary.json", "w", encoding="utf-8") as f:
        json.dump(group_result, f, indent=2)
    print("Group-aware experiment failed:", exc)


,seed,model,n_test,n_features,accuracy,precision,recall,specificity,f1,roc_auc,tn,fp,fn,tp
0,42,Baseline RF (group-aware split),6736,512,0.980552,0.962684,0.997817,0.964873,0.979936,0.999685,3406,124,7,3199
1,42,"RF + BGWO (252 features, group-aware split)",6736,252,0.997328,0.997193,0.997193,0.997450,0.997193,0.999543,3521,9,9,3197
2,42,RF + BGWO top-13 (group-aware split),6736,13,0.986045,0.975260,0.995945,0.977054,0.985494,0.997945,3449,81,13,3193


{
  "group_train_rows": 16123,
  "group_test_rows": 6736,
  "fit_rows": 13033,
  "validation_rows": 3090,
  "template_overlap_train_test": 0,
  "template_overlap_fit_val": 0,
  "bgwo_selected_features": 252,
  "bgwo_best_fitness": 0.005242263349514614,
  "top13_features": [
    "select",
    "select from",
    "union",
    "and",
    "where",
    "case when",
    "1 where",
    "1 select",
    "1 or",
    "delete from",
    "0",
    "1 as",
    "s from"
  ],
  "top13_importances": [
    0.1989059357188326,
    0.09177979170933158,
    0.06513873263421284,
    0.05575116093987357,
    0.04145570138141351,
    0.02680099406590463,
    0.025341447031249287,
    0.02531435414803342,
    0.025019179522045135,
    0.020553913363056446,
    0.017615496858913518,
    0.017178223040693365,
    0.015994298920578628
  ]
}


## 9. False positives/negatives by heuristic attack family

The dataset has no ground-truth attack-family labels. To satisfy the checklist in spirit, a transparent keyword heuristic assigns each injection query to a proxy family (time-based, error/out-of-band, union-based, stacked, boolean/tautology, comment/obfuscation, other). False negatives are counted per proxy family for seed 42. False-positive and false-negative examples are exported with truncated queries. These families are indicative only and must not be reported as ground-truth categories.


In [10]:

def attack_family(query):
    s = str(query).lower()
    if re.search(r"\b(sleep|pg_sleep|benchmark|waitfor\s+delay)\b", s):
        return "time-based"
    if re.search(r"\b(extractvalue|updatexml|utl_inaddr|ctxsys|dbms_|load_file)\b", s):
        return "error/out-of-band"
    if re.search(r"into\s+(out|dump)file", s):
        return "error/out-of-band"
    if re.search(r"\bunion\b", s) and re.search(r"\bselect\b", s):
        return "union-based"
    if re.search(r";\s*(insert|update|delete|drop|create|alter|exec|truncate)\b", s):
        return "stacked"
    if re.search(r"\b(or|and)\b.{0,60}(=|<|>|\blike\b|\bis\b)", s):
        return "boolean/tautology"
    if "--" in s or "#" in s:
        return "comment/obfuscation"
    return "other"


if seed_audit is None:
    raise RuntimeError("Seed 42 predictions were not captured.")

y_test42 = seed_audit["y_test"]
text_test42 = seed_audit["text_test"]
top13_pred42 = seed_audit["top13_pred"]
baseline_pred42 = seed_audit["baseline_pred"]

inj_mask = y_test42 == 1
family_series = pd.Series([attack_family(q) for q in text_test42[inj_mask]])
family_totals = (
    family_series.value_counts()
    .rename_axis("attack_family_proxy")
    .reset_index(name="injections")
)
family_totals["top13_missed"] = (
    family_series[top13_pred42[inj_mask] == 0]
    .value_counts()
    .reindex(family_totals["attack_family_proxy"], fill_value=0)
    .to_numpy()
)
family_totals["top13_fn_rate_pct"] = (
    100 * family_totals["top13_missed"] / family_totals["injections"]
)
family_totals["baseline_missed"] = (
    family_series[baseline_pred42[inj_mask] == 0]
    .value_counts()
    .reindex(family_totals["attack_family_proxy"], fill_value=0)
    .to_numpy()
)
family_totals["baseline_fn_rate_pct"] = (
    100 * family_totals["baseline_missed"] / family_totals["injections"]
)
family_totals = family_totals.sort_values("injections", ascending=False).reset_index(drop=True)
family_totals.to_csv(ARTIFACT_DIR / "attack_family_fn.csv", index=False)
display(family_totals)

fp_mask = (y_test42 == 0) & (top13_pred42 == 1)
fn_mask = (y_test42 == 1) & (top13_pred42 == 0)
examples = pd.DataFrame({
    "error_type": (["false_positive"] * int(fp_mask.sum()))
    + (["false_negative"] * int(fn_mask.sum())),
    "proxy_family": (["benign"] * int(fp_mask.sum()))
    + ([attack_family(q) for q in text_test42[fn_mask]]),
    "query_excerpt": [str(q)[:160] for q in text_test42[fp_mask]]
    + [str(q)[:160] for q in text_test42[fn_mask]],
})
examples.to_csv(ARTIFACT_DIR / "fp_fn_examples.csv", index=False)
display(examples.head(20))
print(
    f"Seed 42 top-13 model: FP={int(fp_mask.sum())}, FN={int(fn_mask.sum())}, "
    f"test rows={len(y_test42)}"
)


,attack_family_proxy,injections,top13_missed,top13_fn_rate_pct,baseline_missed,baseline_fn_rate_pct
0,boolean/tautology,1353,0,0.000000,0,0.000000
1,union-based,661,0,0.000000,0,0.000000
2,time-based,499,3,0.601202,0,0.000000
3,other,391,4,1.023018,4,1.023018
4,comment/obfuscation,311,3,0.964630,4,1.286174
5,error/out-of-band,165,2,1.212121,0,0.000000
6,stacked,4,0,0.000000,0,0.000000


,error_type,proxy_family,query_excerpt
0,false_positive,benign,"INSERT INTO up ( simple, radio, act, combinati..."
1,false_positive,benign,"INSERT INTO actually ( sides, year, recognize..."
2,false_positive,benign,"INSERT INTO underline ( herself, dull, social..."
3,false_positive,benign,"INSERT INTO with ( leaf, trip, today, life, vi..."
4,false_positive,benign,"INSERT INTO saw ( knowledge, at, manufacturing..."
5,false_positive,benign,"UPDATE call SET member = 'fifteen', City = '..."
6,false_positive,benign,"SELECT STR_TO_DATE ( ""2017,8,14 10,40,10"", ""%Y..."
7,false_positive,benign,"com"" style = ""max-width:450px;float:left;pad..."
8,false_positive,benign,"INSERT INTO move ( try, think, within ) VALU..."
9,false_positive,benign,"INSERT INTO pick ( tip, could, specific, far, ..."


Seed 42 top-13 model: FP=88, FN=12, test rows=6858


## 10. Checklist summary and saved artifacts

Each publication-checklist item is mapped to its status and evidence. Items requiring resources that are not available (an external dataset) are marked explicitly rather than silently skipped.


In [11]:

checklist = pd.DataFrame([
    {
        "item": "Clean-kernel execution and saved outputs",
        "status": "Done",
        "evidence": "Notebook executed end-to-end with nbconvert; all code cells completed without errors.",
    },
    {
        "item": "Label mapping and class counts before/after cleaning",
        "status": "Done",
        "evidence": (
            f"Raw rows={audit['raw_rows']}; final rows={audit['rows_after_duplicate_removal']}; "
            "0=Benign, 1=SQL injection; counts saved in dataset_audit.json."
        ),
    },
    {
        "item": "Duplicate/template leakage investigation",
        "status": "Done",
        "evidence": (
            f"Exact duplicate queries remaining={audit['exact_duplicate_queries_remaining']}; "
            f"random-split test template overlap={leakage['random_split_test_template_overlap_pct']:.2f}%; "
            "group-aware results saved in group_split_metrics.csv."
        ),
    },
    {
        "item": "Repeat the full process across multiple seeds",
        "status": "Done",
        "evidence": (
            f"Seeds={SEEDS}; mean/std and 95% CIs saved in multiseed_summary.csv."
        ),
    },
    {
        "item": "External or cross-dataset test",
        "status": "Not available",
        "evidence": "No external dataset was provided; generalization claims require one.",
    },
    {
        "item": "Text baselines with the same split",
        "status": "Done",
        "evidence": (
            "Logistic Regression and Linear SVM on word and char(3-5) TF-IDF "
            "were evaluated per seed in multiseed_metrics.csv."
        ),
    },
    {
        "item": "False positives/negatives by attack family",
        "status": "Done (proxy)",
        "evidence": (
            "No ground-truth family labels exist; heuristic keyword families and "
            "examples are saved in attack_family_fn.csv and fp_fn_examples.csv."
        ),
    },
    {
        "item": "Environment and runtime details",
        "status": "Done",
        "evidence": "environment.json and sklearn_versions.txt saved.",
    },
    {
        "item": "Treat the test set as consumed; no tuning on test",
        "status": "Done",
        "evidence": (
            "TF-IDF, BGWO, and top-13 ranking use fit/validation data only; "
            "test partitions are used for final scoring."
        ),
    },
    {
        "item": "No production-readiness claim",
        "status": "Note",
        "evidence": (
            "Results are experimental; parameterized queries remain the primary defense."
        ),
    },
    {
        "item": "Top-13 ranking computed on training/fit data only",
        "status": "Done",
        "evidence": "ranking_rf is fitted on X_fit/y_fit; test labels are never used.",
    },
])
checklist.to_csv(ARTIFACT_DIR / "checklist_summary.csv", index=False)
display(checklist)

metadata = {
    "dataset_path": str(DATA_PATH.resolve()),
    "seeds": SEEDS,
    "bgwo_population": BGWO_POPULATION,
    "bgwo_iterations": BGWO_ITERATIONS,
    "fitness_rf_trees": FITNESS_RF_TREES,
    "final_rf_trees": RF_TREES,
    "word_max_features": MAX_FEATURES,
    "word_ngram_range": list(NGRAM_RANGE),
    "char_max_features": CHAR_MAX_FEATURES,
    "top_k": TOP_K,
    "audit": audit,
    "template_leakage": leakage,
    "group_split": group_result,
    "environment": env,
}
with open(ARTIFACT_DIR / "publication_checks_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Publication-check artifacts saved to:", ARTIFACT_DIR.resolve())
print("Files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name)


,item,status,evidence
0,Clean-kernel execution and saved outputs,Done,Notebook executed end-to-end with nbconvert; a...
1,Label mapping and class counts before/after cl...,Done,"Raw rows=22916; final rows=22859; 0=Benign, 1=..."
2,Duplicate/template leakage investigation,Done,Exact duplicate queries remaining=0; random-sp...
3,Repeat the full process across multiple seeds,Done,"Seeds=[42, 43, 44, 45, 46]; mean/std and 95% C..."
4,External or cross-dataset test,Not available,No external dataset was provided; generalizati...
5,Text baselines with the same split,Done,Logistic Regression and Linear SVM on word and...
6,False positives/negatives by attack family,Done (proxy),No ground-truth family labels exist; heuristic...
7,Environment and runtime details,Done,environment.json and sklearn_versions.txt saved.
8,Treat the test set as consumed; no tuning on test,Done,"TF-IDF, BGWO, and top-13 ranking use fit/valid..."
9,No production-readiness claim,Note,Results are experimental; parameterized querie...


Publication-check artifacts saved to: C:\Users\user\Desktop\fix leake\publication_checks_artifacts
Files:
 - attack_family_fn.csv
 - bgwo_history_group_split.csv
 - bgwo_history_seed_42.csv
 - bgwo_history_seed_43.csv
 - bgwo_history_seed_44.csv
 - bgwo_history_seed_45.csv
 - bgwo_history_seed_46.csv
 - checklist_summary.csv
 - dataset_audit.json
 - environment.json
 - fp_fn_examples.csv
 - group_split_metrics.csv
 - group_split_summary.json
 - multiseed_metrics.csv
 - multiseed_run_metadata.csv
 - multiseed_summary.csv
 - publication_checks_metadata.json
 - sklearn_versions.txt
 - template_leakage.json
 - top13_stability.csv
